In [1]:
adhoc: str
#adhoc = adhoc.strip("'\"")
#print(f"cleaned adhoc: >{adhoc}<")

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 3, Finished, Available, Finished, False)

start_date = '2025-04-01'
end_date = '2025-04-30'

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import json 
from pyspark.sql.functions import current_timestamp, trunc, add_months, date_format, col
import pytz
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta # added this for datetime calculation
from pyspark.sql.functions import col, explode
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from pyspark.sql.functions import col, lit, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType , DateType , BooleanType , DoubleType ,TimestampType,ArrayType,ArrayType,LongType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType
spark = SparkSession.builder.appName("json-to-parquet").config("spark.driver.maxResultSize", "-1").getOrCreate()
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
import pandas as pd
import pyodbc

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 4, Finished, Available, Finished, False)

In [3]:
# Get current time in Eastern Time
eastern = pytz.timezone("US/Eastern")
now_et = datetime.now(eastern)
# now_et = now_et - relativedelta(months=1)

# First day of this month (exclusive upper bound) -- to ensure end date be first of current month and hence avoiding missing edge cases 
end_date_prev_month = now_et.replace(day=1)

# First day of previous month -- to ensure start date be first of previous month and hence avoiding missing edge cases
start_date_prev_month = end_date_prev_month - relativedelta(months=1)
  

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 5, Finished, Available, Finished, False)

In [4]:
eastern = pytz.timezone("US/Eastern")
current_datetime = datetime.now(eastern)

#current_datetime=datetime.now()
# Split into date and time
rundate = current_datetime.date()
runtime = current_datetime.time()

print("Run date:", rundate)
print("Run time:", runtime)

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 6, Finished, Available, Finished, False)

Run date: 2026-04-14
Run time: 11:29:21.934716


adhoc = 'No'

In [6]:
#In line with keeping our start_date to first of month and end date to start of next month 
if adhoc == 'No':
    #print(f"adhoc raw value: >{adhoc}<")
    start_date = start_date_prev_month.strftime('%Y-%m-%d')
    end_date = end_date_prev_month.strftime('%Y-%m-%d')
else:
    #print(f"adhoc raw value: >{adhoc}<")
    start_date = now_et.replace(day=1).strftime('%Y-%m-%d')
    end_date=str(rundate)
print(start_date) # if not Adhoc should look like 2026-02-01
print(end_date) # if not Adhoc should look like 2026-03-01

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 8, Finished, Available, Finished, False)

2026-03-01
2026-04-01


In [7]:
# start_date = '2025-07-30'
# end_date = '2025-09-01'

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 9, Finished, Available, Finished, False)

!pip list | grep pyodbc

!nc -zv ctsus10db01.database.windows.net 1433


In [8]:
#spark.catalog.clearCache()
#spark.sql("REFRESH TABLE UAT_DEBIH.PR1_STATEMENT_NEXT_FEE_Q")
#spark.sql("REFRESH TABLE UAT_DEBIH.PR1_STATEMENT_FEE")
#spark.sql("REFRESH TABLE UAT_DEBIH.PR1_ACCOUNTP")

PolicyTotalPremium = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/TransactionTotalPremium_versions' # Replaced 'TotalPremium_versions' with 'TransactionTotalPremium_versions' for reconcilliation effort with SR23217A
PolicyTransactions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Transaction_versions'
Policy_versions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Policy_versions'
policy_fees = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/FeesAndTaxes_versions'
#Pr1_STATEMENT_NEXT_FEE_Q = 'abfss://UAT_IHv2@onelake.dfs.fabric.microsoft.com/UAT_DEBIH.Lakehouse/Tables/PR1_STATEMENT_NEXT_FEE_Q'
#Pr1_STATEMENT_FEE = 'abfss://UAT_IHv2@onelake.dfs.fabric.microsoft.com/UAT_DEBIH.Lakehouse/Tables/PR1_STATEMENT_FEE'
#pr1_accountp = 'abfss://UAT_IHv2@onelake.dfs.fabric.microsoft.com/UAT_DEBIH.Lakehouse/Tables/PR1_ACCOUNTP'

# Read data into DataFrames
PolicyTotalPremium = spark.read.format("delta").load(PolicyTotalPremium)
PolicyTransactions=spark.read.format("delta").load(PolicyTransactions)
Policy_versions=spark.read.format("delta").load(Policy_versions)
policy_fees=spark.read.format("delta").load(policy_fees)
#Pr1_STATEMENT_NEXT_FEE_Q=spark.read.format("delta").load(Pr1_STATEMENT_NEXT_FEE_Q)
#Pr1_STATEMENT_FEE=spark.read.format("delta").load(Pr1_STATEMENT_FEE)
#pr1_accountp=spark.read.format("delta").load(pr1_accountp)

#pr1_accountp.printSchema()

spark.catalog.dropTempView("PolicyTotalPremium")
spark.catalog.dropTempView("PolicyTransactions")
spark.catalog.dropTempView("Policy_versions")
spark.catalog.dropTempView("policy_fees")
#spark.catalog.dropTempView("Pr1_STATEMENT_NEXT_FEE_Q")
#spark.catalog.dropTempView("Pr1_STATEMENT_FEE")
#spark.catalog.dropTempView("pr1_accountp")


Pr1_STATEMENT_NEXT_FEE_Q = spark.sql("select * from PROD_DEBIH.PR1_STATEMENT_NEXT_FEE_Q")
Pr1_STATEMENT_FEE = spark.sql("select * from PROD_DEBIH.PR1_STATEMENT_FEE")
pr1_accountp = spark.sql("select * from PROD_DEBIH.PR1_ACCOUNTP")

# Create temporary views
PolicyTotalPremium.createOrReplaceTempView("PolicyTotalPremium")
PolicyTransactions.createOrReplaceTempView("PolicyTransactions")
Policy_versions.createOrReplaceTempView("Policy_versions")
policy_fees.createOrReplaceTempView("policy_fees")
Pr1_STATEMENT_NEXT_FEE_Q.createOrReplaceTempView("Pr1_STATEMENT_NEXT_FEE_Q")
Pr1_STATEMENT_FEE.createOrReplaceTempView("Pr1_STATEMENT_FEE")
pr1_accountp.createOrReplaceTempView("pr1_accountp")


StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 10, Finished, Available, Finished, False)

In [9]:
spark.catalog.clearCache()

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 11, Finished, Available, Finished, False)

In [10]:
# Original Query broken down for testing
# query = f'''select 'ALP Issued Policies Transaction Register' as ReportPageName,
# date_format('{rundate}','MM/dd/yyyy') as rundate,
# SUBSTRING('{runtime}',1,8) as runtime,
# date_format('{start_date}','MM/dd/yyyy') as startdate, 
# date_format('{end_date}','MM/dd/yyyy') as enddate,
# SUBSTRING(p.policynumber,1,4) as policytype,SUBSTRING(p.policynumber,5,len(p.policynumber)) as policynumber,
# date_format(from_utc_timestamp(t.effectivedate,'EST'),'MM/dd/yyyy') as transaction_eff_date,
# CASE WHEN t.Status='Committed' THEN
# 		   CASE WHEN t.type in ('Policy','NEW') THEN 'New Business' 
# 		  		WHEN t.type = 'Cancellation' THEN CONCAT(t.type,' - ',t.PremiumType)
# 				ELSE t.type END
# 	 ELSE 
# 	 	CASE WHEN  p.PolicyStatus = 'Non-Renewal - Added' THEN 'Non-Renewal' ELSE p.PolicyStatus END		  
# 	END as transaction_type,
# CASE WHEN from_utc_timestamp(t.effectivedate,'EST') < '{end_date}' then 'Current'
# 	 WHEN from_utc_timestamp(t.effectivedate,'EST') >= '{end_date}' then 'Future'
# 	 END as effective_type,
# CASE WHEN t.Status='Voided' THEN 0.00 
# 	 WHEN t.type='Cancellation' THEN -1*prem.annualpremium
# 	 WHEN t.type='Endorsement' THEN prem.annualpremium - prem.priorannualpremium
# 	 ELSE prem.annualpremium END as inforce_prem,
# CASE WHEN t.Status='Voided' THEN 0.00 ELSE prem.effectivepremium END as written_prem,
# CASE WHEN t.Status='Voided' THEN 0.00
# 	 WHEN t.type='Endorsement' THEN 0.00
# 	 WHEN t.type='Cancellation' THEN -1*fees.inforce_fees
# 	 ELSE fees.inforce_fees END as inforce_fees,
# CASE WHEN t.Status='Voided' THEN 0.00
# 	 WHEN t.type='Endorsement' THEN 0.00 
# 	 ELSE fees.written_fees END as written_fees,
# 0.00 as taxes,
# 0.00 as misc

# from policy_versions p
# join PolicyTransactions t
# on t.policy_ref=p.policy_ref
# join PolicyTotalPremium prem
# on prem.policy_ref=p.policy_ref
# join 
# (select policy_ref,sum(Amount) as written_fees,sum(AnnualAmount) as inforce_fees from policy_fees group by policy_ref) fees
# on fees.policy_ref = t.policy_ref
# where ((t.status = 'Committed') OR 
# 	(t.Status='Voided' and t.type = 'Cancellation' and p.PolicyStatus = 'Pending Cancellation') OR
# 	(t.Status='Voided' and t.type='Renewal' and  p.PolicyStatus = 'Renewal-Offered') OR
# 	(t.Status='Voided' and t.type='Reinstate' and  p.PolicyStatus = 'Rescind') OR
# 	(t.Status='Voided' and  p.PolicyStatus = 'Non-Renewal - Added')
# 	)
# --and from_utc_timestamp(t.date,'EST') between '{start_date}' and '{end_date}'
# and t.date >= '{start_date}' and  t.date < '{end_date}'
# --and p.PolicyNumber='

# union all

#  -- billing fees (late fee, nsf fee, reinstate fee)
# 	select 'Non-ALP Issued Policies Transaction Register' as ReportPageName, 
# 	date_format('{rundate}','MM/dd/yyyy') as rundate,
# 	SUBSTRING('{runtime}',1,8) as runtime,
# 	date_format('{start_date}','MM/dd/yyyy') as startdate, 
# 	date_format('{end_date}','MM/dd/yyyy') as enddate,
# SUBSTRING(p.accountp_policy_num,1,4) as policytype,SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) as policynumber,
# date_format(fq.SFQ_PAID_DATE,'MM/dd/yyyy') as transaction_eff_date,
# CASE SFQ_FEE_CODE WHEN 5 THEN 'Late Fee'
# 				  WHEN 2 THEN 'NSF Fee'
# 				  WHEN 4 THEN 'Reinstatement Fee' 
# 	END	as transaction_type,
# CASE WHEN fq.SFQ_PAID_DATE >= '{start_date}' and fq.SFQ_PAID_DATE < '{end_date}' then 'Current'
# 	 WHEN fq.SFQ_PAID_DATE > '{end_date}' then 'Future'
# 	 END as effective_type,
# 0 as inforce_prem,
# 0 as written_prem,
# fq.SFQ_FEE_AMOUNT as inforce_fees,
# fq.SFQ_FEE_AMOUNT as written_fees,
# 0 as taxes,
# 0 as misc

# from PR1_STATEMENT_NEXT_FEE_Q fq
# join pr1_accountp p
# on p.accountp_number=fq.SFQ_APPLY_TO_ACCOUNT
# where SFQ_FEE_CODE in(5,2,4)
# and p.accountp_policy_condition=1
# and SFQ_IS_VALID = 1
# and SFQ_IS_WAIVED <> 1
# and fq.SFQ_PAID_DATE >= '{start_date}' and fq.SFQ_PAID_DATE < '{end_date}'

# union all

# -- statement fees (installment and eft fees)
# 	select
# 	'Non-ALP Issued Policies Transaction Register' as ReportPageName, 
# 	date_format('{rundate}','MM/dd/yyyy') as rundate,
# 	SUBSTRING('{runtime}',1,8) as runtime,
# 	date_format('{start_date}','MM/dd/yyyy') as startdate, 
# 	date_format('{end_date}','MM/dd/yyyy') as enddate,
# SUBSTRING(p.accountp_policy_num,1,4) as policytype,SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) as policynumber,
# date_format(f.SF_PAID_DATE,'MM/dd/yyyy') as transaction_eff_date,
# CASE SF_FEE_CODE WHEN 1 THEN 'Installment Fee'
# 				 WHEN 3 THEN 'EFT Fee'
# 	END	as transaction_type,
# CASE WHEN f.SF_PAID_DATE >= '{start_date}' and f.SF_PAID_DATE < '{end_date}' then 'Current'
# 	 WHEN f.SF_PAID_DATE > '{end_date}' then 'Future'
# 	 END as effective_type,
# 0 as inforce_prem,
# 0 as written_prem,
# f.SF_AMOUNT as inforce_fees,
# f.SF_AMOUNT as written_fees,
# 0 as taxes,
# 0 as misc
			
# from PR1_STATEMENT_FEE f
# join pr1_accountp p
# on p.accountp_number=f.SF_STATEMENT_NUMBER
# where SF_FEE_CODE in(1,3)
# and p.accountp_policy_condition=1
# and SF_IS_REVERSED <> 1 
# and SF_IS_WAIVED <> 1
# and SF_IS_VALID = 1
# and SF_PAID_DATE >= '{start_date}' and SF_PAID_DATE < '{end_date}'

#  '''

# # Execute the query
# result = spark.sql(query)
# result.show()

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 12, Finished, Available, Finished, False)

In [11]:
#Original Query broken down for testing
policy_query =f"""
select 'ALP Issued Policies Transaction Register' as ReportPageName,
date_format('{rundate}','MM/dd/yyyy') as rundate,
SUBSTRING('{runtime}',1,8) as runtime,
date_format('{start_date}','MM/dd/yyyy') as startdate, 
date_format('{end_date}','MM/dd/yyyy') as enddate,
SUBSTRING(p.policynumber,1,4) as policytype,SUBSTRING(p.policynumber,5,len(p.policynumber)) as policynumber,
date_format(t.effectivedate,'MM/dd/yyyy') as transaction_eff_date,
CASE WHEN t.Status='Committed' THEN
		   CASE WHEN t.type in ('Policy','NEW') THEN 'New Business' 
		  		WHEN t.type = 'Cancellation' THEN CONCAT(t.type,' - ',t.PremiumType)
				ELSE t.type END
	 ELSE 
	 	CASE WHEN  p.PolicyStatus = 'Non-Renewal - Added' THEN 'Non-Renewal' ELSE p.PolicyStatus END		  
	END as transaction_type,
CASE WHEN t.effectivedate < '{end_date}' then 'Current'
	 WHEN t.effectivedate >= '{end_date}' then 'Future'
	 END as effective_type,
CASE WHEN t.Status='Voided' THEN 0.00 
	 WHEN t.type='Cancellation' THEN -1*prem.annualpremium
	 WHEN t.type='Endorsement' THEN prem.annualpremium - prem.priorannualpremium
	 ELSE prem.annualpremium END as inforce_prem,
CASE WHEN t.Status='Voided' THEN 0.00 ELSE prem.effectivepremium END as written_prem,
CASE WHEN t.Status='Voided' THEN 0.00
	 WHEN t.type='Endorsement' THEN 0.00
	 WHEN t.type='Cancellation' THEN -1*fees.inforce_fees
	 ELSE fees.inforce_fees END as inforce_fees,
CASE WHEN t.Status='Voided' THEN 0.00
	 WHEN t.type='Endorsement' THEN 0.00 
	 ELSE fees.written_fees END as written_fees,
0.00 as taxes,
0.00 as misc

from policy_versions p
join PolicyTransactions t
on t.policy_ref=p.policy_ref
join PolicyTotalPremium prem
on prem.policy_ref=p.policy_ref
join 
(select policy_ref,sum(Amount) as written_fees,sum(AnnualAmount) as inforce_fees from policy_fees group by policy_ref) fees
on fees.policy_ref = t.policy_ref
where ((t.status = 'Committed') OR 
	(t.Status='Voided' and t.type = 'Cancellation' and p.PolicyStatus = 'Pending Cancellation') OR
	(t.Status='Voided' and t.type='Renewal' and  p.PolicyStatus = 'Renewal-Offered') OR
	(t.Status='Voided' and t.type='Reinstate' and  p.PolicyStatus = 'Rescind') OR
	(t.Status='Voided' and  p.PolicyStatus = 'Non-Renewal - Added')
	)
and t.date >= '{start_date}' and  t.date < '{end_date}'
AND p.PolicyNumber = 'GAPA006485106-2'

"""
# Execute the query
result_policy = spark.sql(policy_query)
display(result_policy)

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e2292487-9421-466f-8161-599f48e05e49)

In [12]:
# Original Query broken down for testing
# billing_query1 = f"""
#  -- billing fees (late fee, nsf fee, reinstate fee)
# 	select 'Non-ALP Issued Policies Transaction Register' as ReportPageName, 
# 	date_format('{rundate}','MM/dd/yyyy') as rundate,
# 	SUBSTRING('{runtime}',1,8) as runtime,
# 	date_format('{start_date}','MM/dd/yyyy') as startdate, 
# 	date_format('{end_date}','MM/dd/yyyy') as enddate,
# SUBSTRING(p.accountp_policy_num,1,4) as policytype,SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) as policynumber,
# date_format(fq.SFQ_PAID_DATE,'MM/dd/yyyy') as transaction_eff_date,
# CASE SFQ_FEE_CODE WHEN 5 THEN 'Late Fee'
# 				  WHEN 2 THEN 'NSF Fee'
# 				  WHEN 4 THEN 'Reinstatement Fee' 
# 	END	as transaction_type,
# CASE WHEN fq.SFQ_PAID_DATE >= '{start_date}' and fq.SFQ_PAID_DATE < '{end_date}' then 'Current'
# 	 WHEN fq.SFQ_PAID_DATE > '{end_date}' then 'Future'
# 	 END as effective_type,
# 0 as inforce_prem,
# 0 as written_prem,
# fq.SFQ_FEE_AMOUNT as inforce_fees,
# fq.SFQ_FEE_AMOUNT as written_fees,
# 0 as taxes,
# 0 as misc

# from PR1_STATEMENT_NEXT_FEE_Q fq
# join pr1_accountp p
# on p.accountp_number=fq.SFQ_APPLY_TO_ACCOUNT
# where SFQ_FEE_CODE in(5,2,4)
# and p.accountp_policy_condition=1
# and SFQ_IS_VALID = 1
# and SFQ_IS_WAIVED <> 1
# and fq.SFQ_PAID_DATE >= '{start_date}' and fq.SFQ_PAID_DATE < '{end_date}'

# """
# b1_result = spark.sql(billing_query1)
# display(b1_result)

StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 14, Finished, Available, Finished, False)

In [13]:
# Original Query broken down for testing
# b2_q = f"""
# -- statement fees (installment and eft fees)
# 	select
# 	'Non-ALP Issued Policies Transaction Register' as ReportPageName, 
# 	date_format('{rundate}','MM/dd/yyyy') as rundate,
# 	SUBSTRING('{runtime}',1,8) as runtime,
# 	date_format('{start_date}','MM/dd/yyyy') as startdate, 
# 	date_format('{end_date}','MM/dd/yyyy') as enddate,
# SUBSTRING(p.accountp_policy_num,1,4) as policytype,SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) as policynumber,
# date_format(f.SF_PAID_DATE,'MM/dd/yyyy') as transaction_eff_date,
# CASE SF_FEE_CODE WHEN 1 THEN 'Installment Fee'
# 				 WHEN 3 THEN 'EFT Fee'
# 	END	as transaction_type,
# CASE WHEN f.SF_PAID_DATE >= '{start_date}' and f.SF_PAID_DATE < '{end_date}' then 'Current'
# 	 WHEN f.SF_PAID_DATE > '{end_date}' then 'Future'
# 	 END as effective_type,
# 0 as inforce_prem,
# 0 as written_prem,
# f.SF_AMOUNT as inforce_fees,
# f.SF_AMOUNT as written_fees,
# 0 as taxes,
# 0 as misc
			
# from PR1_STATEMENT_FEE f
# join pr1_accountp p
# on p.accountp_number=f.SF_STATEMENT_NUMBER
# where SF_FEE_CODE in(1,3)
# and p.accountp_policy_condition=1
# and SF_IS_REVERSED <> 1 
# and SF_IS_WAIVED <> 1
# and SF_IS_VALID = 1
# and SF_PAID_DATE >= '{start_date}' and SF_PAID_DATE < '{end_date}'

# """
# r2_b =spark.sql(b2_q)
# display(r2_b)


StatementMeta(, 1ec4de21-047c-4774-906e-6e0c26a8cbb3, 15, Finished, Available, Finished, False)

In [14]:
# Original query with Date Comparision changes like between replaced by >= and < ; NOT resulting to THE FINAL OUTPUT
query = f''' With base as(
select 'ALP Issued Policies Transaction Register' as ReportPageName,
date_format('{rundate}','MM/dd/yyyy') as rundate,
SUBSTRING('{runtime}',1,8) as runtime,
date_format('{start_date}','MM/dd/yyyy') as startdate, 
date_format('{end_date}','MM/dd/yyyy') as enddate,
SUBSTRING(p.policynumber,1,4) as policytype,SUBSTRING(p.policynumber,5,len(p.policynumber)) as policynumber,
date_format(from_utc_timestamp(t.effectivedate,'EST'),'MM/dd/yyyy') as transaction_eff_date,
CASE WHEN t.Status='Committed' THEN
		CASE WHEN t.type in ('Policy','NEW') THEN 'New Business' 
				WHEN t.type = 'Cancellation' THEN CONCAT(t.type,' - ',t.PremiumType)
				ELSE t.type END
	ELSE 
		CASE WHEN  p.PolicyStatus = 'Non-Renewal - Added' THEN 'Non-Renewal' ELSE p.PolicyStatus END		  
	END as transaction_type,
CASE WHEN from_utc_timestamp(t.effectivedate,'EST') < '{end_date}' then 'Current'
	WHEN from_utc_timestamp(t.effectivedate,'EST') >= '{end_date}' then 'Future'
	END as effective_type,
CASE WHEN t.Status='Voided' THEN 0.00 
	WHEN t.type='Cancellation' THEN -1*prem.annualpremium
	ELSE prem.annualpremium END as inforce_prem,
CASE WHEN t.Status='Voided' THEN 0.00 ELSE prem.effectivepremium END as written_prem,
CASE WHEN t.Status='Voided' THEN 0.00
	WHEN t.type='Endorsement' THEN 0.00
	WHEN t.type='Cancellation' THEN -1*fees.inforce_fees
	ELSE fees.inforce_fees END as inforce_fees,
CASE WHEN t.Status='Voided' THEN 0.00
	WHEN t.type='Endorsement' THEN 0.00 
	ELSE fees.written_fees END as written_fees,
0.00 as taxes,
0.00 as misc

from policy_versions p
join PolicyTransactions t
on t.policy_ref=p.policy_ref
join PolicyTotalPremium prem
on prem.policy_ref=p.policy_ref
join 
(select policy_ref,sum(Amount) as written_fees,sum(AnnualAmount) as inforce_fees from policy_fees group by policy_ref) fees
on fees.policy_ref = t.policy_ref
where ((t.status = 'Committed') OR
	(t.Status='Voided' and t.type = 'Cancellation' and p.PolicyStatus = 'Pending Cancellation') OR
	(t.Status='Voided' and t.type='Renewal' and  p.PolicyStatus = 'Renewal-Offered') OR
	(t.Status='Voided' and t.type='Reinstate' and  p.PolicyStatus = 'Rescind') OR
	(t.Status='Voided' and  p.PolicyStatus = 'Non-Renewal - Added')
	)
--and from_utc_timestamp(t.date,'EST') between '{start_date}' and '{end_date}'
and t.date >= '{start_date}' and t.date < '{end_date}'
--and p.PolicyNumber='

union all

-- billing fees (late fee, nsf fee, reinstate fee)
	select 'Non-ALP Issued Policies Transaction Register' as ReportPageName, 
	date_format('{rundate}','MM/dd/yyyy') as rundate,
	SUBSTRING('{runtime}',1,8) as runtime,
	date_format('{start_date}','MM/dd/yyyy') as startdate, 
	date_format('{end_date}','MM/dd/yyyy') as enddate,
SUBSTRING(p.accountp_policy_num,1,4) as policytype,SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) as policynumber,
date_format(fq.SFQ_PAID_DATE,'MM/dd/yyyy') as transaction_eff_date,
CASE SFQ_FEE_CODE WHEN 5 THEN 'Late Fee'
				WHEN 2 THEN 'NSF Fee'
				WHEN 4 THEN 'Reinstatement Fee' 
	END	as transaction_type,
CASE WHEN fq.SFQ_PAID_DATE >= '{start_date}' and fq.SFQ_PAID_DATE < '{end_date}' then 'Current'
	WHEN fq.SFQ_PAID_DATE >= '{end_date}' then 'Future'
	END as effective_type,
0 as inforce_prem,
0 as written_prem,
fq.SFQ_FEE_AMOUNT as inforce_fees,
fq.SFQ_FEE_AMOUNT as written_fees,
0 as taxes,
0 as misc

from PR1_STATEMENT_NEXT_FEE_Q fq
join pr1_accountp p
on p.accountp_number=fq.SFQ_APPLY_TO_ACCOUNT
where SFQ_FEE_CODE in(5,2,4)
and p.accountp_policy_condition=1
and SFQ_IS_VALID = 1
and SFQ_IS_WAIVED <> 1
and fq.SFQ_PAID_DATE >= '{start_date}' and fq.SFQ_PAID_DATE < '{end_date}'

union all

-- statement fees (installment and eft fees)
	select
	'Non-ALP Issued Policies Transaction Register' as ReportPageName, 
	date_format('{rundate}','MM/dd/yyyy') as rundate,
	SUBSTRING('{runtime}',1,8) as runtime,
	date_format('{start_date}','MM/dd/yyyy') as startdate, 
	date_format('{end_date}','MM/dd/yyyy') as enddate,
SUBSTRING(p.accountp_policy_num,1,4) as policytype,SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) as policynumber,
date_format(f.SF_PAID_DATE,'MM/dd/yyyy') as transaction_eff_date,
CASE SF_FEE_CODE WHEN 1 THEN 'Installment Fee'
				WHEN 3 THEN 'EFT Fee'
	END	as transaction_type,
CASE WHEN f.SF_PAID_DATE >= '{start_date}' and f.SF_PAID_DATE < '{end_date}' then 'Current'
	WHEN f.SF_PAID_DATE >= '{end_date}' then 'Future'
	END as effective_type,
0 as inforce_prem,
0 as written_prem,
f.SF_AMOUNT as inforce_fees,
f.SF_AMOUNT as written_fees,
0 as taxes,
0 as misc
			
from PR1_STATEMENT_FEE f
join pr1_accountp p
on p.accountp_number=f.SF_STATEMENT_NUMBER
where SF_FEE_CODE in(1,3)
and p.accountp_policy_condition=1
and SF_IS_REVERSED <> 1 
and SF_IS_WAIVED <> 1
and SF_IS_VALID = 1
and SF_PAID_DATE >= '{start_date}' and SF_PAID_DATE < '{end_date}'
) Select Count(*) from base

'''

# Execute the query
result = spark.sql(query)
result.show()

StatementMeta(, 9aa346f2-3936-48d3-9358-d874380d99c9, 16, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|   22267|
+--------+



In [15]:
# Updated Query enabling Multiple month Functionality while avoiding Costly Loops 
changedquery = f"""
--## Generating Start and End Date for last 11 Months ##
WITH months AS ( 
    SELECT
        add_months(date_trunc('month', current_date()), -n) AS start_date,
        add_months(date_trunc('month', current_date()), -n + 1) AS end_date
    FROM (SELECT explode(sequence(0,11)) AS n)
)
--## Policy Data Inforce prem, Written Prem and Fees Calculations ##
, policy_txn AS ( 
SELECT
    'ALP Issued Policies Transaction Register' AS ReportPageName, 
    date_format(current_date(),'MM/dd/yyyy') AS rundate,
    date_format(current_timestamp(),'HH:mm:ss') AS runtime,
    m.start_date,
    m.end_date,
    SUBSTRING(p.policynumber,1,4) AS policytype,
    SUBSTRING(p.policynumber,5,len(p.policynumber)) AS policynumber,
    date_format(t.effectivedate,'MM/dd/yyyy') AS transaction_eff_date,

    CASE
        WHEN t.Status='Committed' THEN
            CASE
                WHEN t.type IN ('Policy','NEW') THEN 'New Business'
                WHEN t.type='Cancellation' THEN CONCAT(t.type,' - ',t.PremiumType)
                ELSE t.type
            END
        ELSE
            CASE
                WHEN p.PolicyStatus='Non-Renewal - Added' THEN 'Non-Renewal'
                ELSE p.PolicyStatus
            END
    END AS transaction_type,

    -- SAME business logic, now using static month boundary
    CASE
        WHEN t.effectivedate < m.end_date THEN 'Current'
        WHEN t.effectivedate >= m.end_date THEN 'Future'
    END AS effective_type,
/*
-- ## Final Bloc updating which above lead to the reconcilliation with SR23217A report ##
    CASE
        WHEN from_utc_timestamp(t.effectivedate,'EST') < m.end_date THEN 'Current'
        WHEN from_utc_timestamp(t.effectivedate,'EST') >= m.end_date THEN 'Future'
    END AS effective_type,
*/
    CASE
        WHEN t.Status='Voided' THEN 0.00
        WHEN t.type='Cancellation' THEN -1 * prem.annualpremium
        ELSE prem.annualpremium
    END AS inforce_prem,

    CASE
        WHEN t.Status='Voided' THEN 0.00
        ELSE prem.effectivepremium
    END AS written_prem,

    CASE
        WHEN t.Status='Voided' THEN 0.00
        WHEN t.type='Endorsement' THEN 0.00
        WHEN t.type='Cancellation' THEN -1 * fees.inforce_fees
        ELSE fees.inforce_fees
    END AS inforce_fees,

    CASE
        WHEN t.Status='Voided' THEN 0.00
        WHEN t.type='Endorsement' THEN 0.00
        ELSE fees.written_fees
    END AS written_fees,
    0.00 AS taxes,
    0.00 AS misc

FROM months m
JOIN PolicyTransactions t
  -- ## SINGLE month ownership (no duplicates) ##
  ON date_trunc('month',t.date) = m.start_date
  --or date_trunc('month', t.effectivedate) = m.start_date

JOIN policy_versions p
  ON p.policy_ref = t.policy_ref
JOIN PolicyTotalPremium prem
  ON prem.policy_ref = p.policy_ref
JOIN (
    SELECT policy_ref,
           SUM(Amount) AS written_fees,
           SUM(AnnualAmount) AS inforce_fees
    FROM policy_fees
    GROUP BY policy_ref
) fees
  ON fees.policy_ref = t.policy_ref

WHERE (
    t.status='Committed'
    OR (t.Status='Voided' AND t.type='Cancellation' AND p.PolicyStatus='Pending Cancellation')
    OR (t.Status='Voided' AND t.type='Renewal' AND p.PolicyStatus='Renewal-Offered')
    OR (t.Status='Voided' AND t.type='Reinstate' AND p.PolicyStatus='Rescind')
    OR (t.Status='Voided' AND p.PolicyStatus='Non-Renewal - Added')
 )
)
--## Billing Fees (Late Fee, NSF Fee, Reinstatement Fee) ##
, billing_fees AS (
    SELECT
        'Non-ALP Issued Policies Transaction Register' AS ReportPageName,
        date_format(current_date(),'MM/dd/yyyy') AS rundate,
        date_format(current_timestamp(),'HH:mm:ss') AS runtime,
        m.start_date,
        m.end_date,
        SUBSTRING(p.accountp_policy_num,1,4) AS policytype,
        SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) AS policynumber,
        date_format(fq.SFQ_PAID_DATE,'MM/dd/yyyy') AS transaction_eff_date,

        CASE fq.SFQ_FEE_CODE
            WHEN 5 THEN 'Late Fee'
            WHEN 2 THEN 'NSF Fee'
            WHEN 4 THEN 'Reinstatement Fee'
        END AS transaction_type,

        CASE
            WHEN fq.SFQ_PAID_DATE >= m.start_date
             AND fq.SFQ_PAID_DATE <  m.end_date THEN 'Current'
            WHEN fq.SFQ_PAID_DATE >= m.end_date THEN 'Future'
        END AS effective_type,

        0 AS inforce_prem,
        0 AS written_prem,
        fq.SFQ_FEE_AMOUNT AS inforce_fees,
        fq.SFQ_FEE_AMOUNT AS written_fees,
        0 AS taxes,
        0 AS misc

    FROM months m
    JOIN PR1_STATEMENT_NEXT_FEE_Q fq
    ON date_trunc('month', fq.SFQ_PAID_DATE) = m.start_date

    JOIN pr1_accountp p
      ON p.accountp_number = fq.SFQ_APPLY_TO_ACCOUNT

    WHERE fq.SFQ_FEE_CODE IN (5,2,4)
      AND p.accountp_policy_condition = 1
      AND fq.SFQ_IS_VALID = 1
      AND fq.SFQ_IS_WAIVED <> 1
)
--## Billing Fees (Installment Fee, EFT Fee) ##
, statement_fees AS (
    SELECT
        'Non-ALP Issued Policies Transaction Register' AS ReportPageName,
        date_format(current_date(),'MM/dd/yyyy') AS rundate,
        date_format(current_timestamp(),'HH:mm:ss') AS runtime,
        m.start_date,
        m.end_date,
        SUBSTRING(p.accountp_policy_num,1,4) AS policytype,
        SUBSTRING(p.accountp_policy_num,5,len(p.accountp_policy_num)) AS policynumber,
        date_format(f.SF_PAID_DATE,'MM/dd/yyyy') AS transaction_eff_date,

        CASE f.SF_FEE_CODE
            WHEN 1 THEN 'Installment Fee'
            WHEN 3 THEN 'EFT Fee'
        END AS transaction_type,

        CASE
            WHEN f.SF_PAID_DATE >= m.start_date
             AND f.SF_PAID_DATE <  m.end_date THEN 'Current'
            WHEN f.SF_PAID_DATE >= m.end_date THEN 'Future'
        END AS effective_type,

        0 AS inforce_prem,
        0 AS written_prem,
        f.SF_AMOUNT AS inforce_fees,
        f.SF_AMOUNT AS written_fees,
        0 AS taxes,
        0 AS misc

    FROM months m
    JOIN PR1_STATEMENT_FEE f
    ON date_trunc('month', f.SF_PAID_DATE) = m.start_date


    JOIN pr1_accountp p
      ON p.accountp_number = f.SF_STATEMENT_NUMBER

    WHERE f.SF_FEE_CODE IN (1,3)
      AND f.SF_IS_REVERSED <> 1
      AND f.SF_IS_WAIVED <> 1
      AND f.SF_IS_VALID = 1
      AND p.accountp_policy_condition = 1
) 
SELECT * FROM policy_txn
--where policynumber = '006757789'
UNION ALL
SELECT * FROM billing_fees
--where policynumber = '006757789'
UNION ALL
SELECT * FROM statement_fees
--where policynumber = '006757789'

 """

# Execute the query
app = spark.sql(changedquery)
app.show(100)


StatementMeta(, 9aa346f2-3936-48d3-9358-d874380d99c9, 17, Finished, Available, Finished, False)

+--------------------+----------+--------+----------+----------+----------+------------+--------------------+----------------+--------------+------------+------------+------------+------------+-----+----+
|      ReportPageName|   rundate| runtime|start_date|  end_date|policytype|policynumber|transaction_eff_date|transaction_type|effective_type|inforce_prem|written_prem|inforce_fees|written_fees|taxes|misc|
+--------------------+----------+--------+----------+----------+----------+------------+--------------------+----------------+--------------+------------+------------+------------+------------+-----+----+
|ALP Issued Polici...|04/09/2026|21:59:51|2026-03-01|2026-04-01|      GAPA|   006757789|          03/17/2026|    New Business|       Current|     1118.00|      1118.0|       30.00|        30.0| 0.00|0.00|
|ALP Issued Polici...|04/09/2026|21:59:51|2026-04-01|2026-05-01|      GAPA|   006757789|          03/27/2026|     Endorsement|       Current|     1348.00|      217.58|        0.00|

In [16]:
# Convert all numeric columns in Spark
app = app.withColumn("inforce_prem", col("inforce_prem").cast("double")) \
         .withColumn("written_prem", col("written_prem").cast("double")) \
         .withColumn("inforce_fees", col("inforce_fees").cast("double")) \
         .withColumn("written_fees", col("written_fees").cast("double")) \
         .withColumn("taxes", col("taxes").cast("double")) \
         .withColumn("misc", col("misc").cast("double"))

# Then convert to pandas
df = app.toPandas()

StatementMeta(, f5388278-f865-482a-8a73-5cc65db16926, 18, Finished, Available, Finished, False)

In [17]:
schema  = StructType([
StructField("ReportPageName",StringType(), True),
StructField("rundate",StringType(), True),
StructField("runtime",StringType(), True),
StructField("startdate",DateType(), True),
StructField("enddate",DateType(), True),
StructField("policytype", StringType(), True),
StructField("policynumber", StringType(), True),
StructField("transaction_eff_date", StringType(), True),
StructField("transaction_type", StringType(), True),
StructField("effective_type", StringType(), True),
StructField("inforce_prem", DoubleType(), True),
StructField("written_prem", DoubleType(), True),
StructField("inforce_fees", DoubleType(), True),
StructField("written_fees", DoubleType(), True),
StructField("taxes", DoubleType(), True),
StructField("misc", DoubleType(), True),
])  
try:
    
    df_spark=spark.createDataFrame(df,schema)
except:

    df_spark=spark.createDataFrame([],schema)
df_spark.show()
df_spark.dtypes    

StatementMeta(, f5388278-f865-482a-8a73-5cc65db16926, 19, Finished, Available, Finished, False)

+--------------------+----------+--------+----------+----------+----------+------------+--------------------+--------------------+--------------+------------+------------+------------+------------+-----+----+
|      ReportPageName|   rundate| runtime| startdate|   enddate|policytype|policynumber|transaction_eff_date|    transaction_type|effective_type|inforce_prem|written_prem|inforce_fees|written_fees|taxes|misc|
+--------------------+----------+--------+----------+----------+----------+------------+--------------------+--------------------+--------------+------------+------------+------------+------------+-----+----+
|ALP Issued Polici...|04/02/2026|21:11:44|2025-10-01|2025-11-01|      GAPA|   006750342|          10/07/2025|Cancellation - Pr...|       Current|     -1196.0|    -1030.96|       -30.0|      -25.86|  0.0| 0.0|
|ALP Issued Polici...|04/02/2026|21:11:44|2025-12-01|2026-01-01|      GAPA|   006751926|          12/02/2025|Cancellation - Pr...|       Current|     -1060.0|     -

[('ReportPageName', 'string'),
 ('rundate', 'string'),
 ('runtime', 'string'),
 ('startdate', 'date'),
 ('enddate', 'date'),
 ('policytype', 'string'),
 ('policynumber', 'string'),
 ('transaction_eff_date', 'string'),
 ('transaction_type', 'string'),
 ('effective_type', 'string'),
 ('inforce_prem', 'double'),
 ('written_prem', 'double'),
 ('inforce_fees', 'double'),
 ('written_fees', 'double'),
 ('taxes', 'double'),
 ('misc', 'double')]

In [18]:
df_spark.write.mode('overwrite').format('delta').option("overwriteSchema", "true").save('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/AL00085_1D')

StatementMeta(, f5388278-f865-482a-8a73-5cc65db16926, 20, Finished, Available, Finished, False)